In [6]:
import os
import geopandas as gpd
import pandas as pd
import numpy as np
from glob import glob
import rasterio as rio
from rasterio.mask import mask
from rasterio.io import MemoryFile
from rasterio.plot import show
from PIL import Image
import json
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from threading import Lock
from src.mslandcover.config import MSTM_PROJ4, LEGEND_CLASSES
from src.mslandcover.utils import raise_if_not_exists
import cv2
import matplotlib.pyplot as plt
from multiprocessing import Pool
from tqdm import tqdm

from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from os import cpu_count

In [5]:
# load the shapefilse with boundaries of the regions
shapefiles = glob(r'Z:\guser\dh\NAIP_MS_2023\*\*.shp')
gdfs = []
for shapefile in shapefiles:
    gdf = gpd.read_file(shapefile).to_crs(MSTM_PROJ4) # convert to the same projection
    raster_path = glob(os.path.join(os.path.dirname(shapefile), '*_1m.tif'))[0]
    gdf['raster_path'] = raster_path
    gdfs.append(gdf)

raster_boundaries_gdf = gpd.GeoDataFrame(pd.concat(gdfs))[['raster_path', 'geometry']] # only keep the relevant columns
raster_boundaries_gdf = raster_boundaries_gdf.dissolve(by='raster_path').reset_index() # dissolve the geometries to get the boundaries of the raster
raster_boundaries_gdf.to_file('data/sampling/regions_boundaries.gpkg', driver='GPKG')

KeyboardInterrupt: 

In [7]:
# load the samples parquet
raster_boundaries_gdf = gpd.read_file('data/sampling/regions_boundaries.gpkg')
samples = gpd.read_parquet('./data/sampling/samples.par')

# spatial join with the raster boundaries
raster_boundaries_gdf['raster_geometry'] = raster_boundaries_gdf['geometry'] # make a copy of the geometry
samples = gpd.sjoin(samples, raster_boundaries_gdf, predicate='intersects', how='left')

In [ ]:
def extract_mask(sample, raster_dataset):
    
    try:
        out_image, out_transform = mask(raster_dataset, [sample['geometry']], crop=True, all_touched=True)
    except Exception as e:
        print(f'type(e) raised while extracting mask for sample {sample.name} in split {sample['split']}: {e}')
        return
    out_meta = raster_dataset.meta.copy()

    if out_image.shape[1] > 256 or out_image.shape[2] > 256:
        # crop the image to 256x256
        out_image = out_image[:, :256, :256]

    filename = str(sample.name)
    if out_image.shape != (3, 256, 256):\
        return
    
    nd_values = np.array([raster_dataset.nodata] * 3)
    pixels_with_nd = np.equal(out_image.transpose(1, 2, 0).reshape(-1, 3), nd_values).all(axis=1)
    if pixels_with_nd.any():
        
        if pixels_with_nd.sum() > 0.05 * len(pixels_with_nd):
            return
    
    out_meta.update({
        'driver': 'GTiff',
        'height': out_image.shape[1],
        'width': out_image.shape[2],
        'transform': out_transform,
    })
    
    # need to segment and convert imgery to polygons for annotation later,
    # having Null nodata values makes this process easier
    if sample['split'] in ('train', 'val', 'test'):
        out_meta['nodata'] = None
    
    out_path = os.path.join('data', 'splits', sample['split'], 'input', filename + '.tif')
    with rio.open(out_path, 'w', **out_meta) as dst:
        dst.write(out_image)
    
    if sample['split'] == 'pretrain' or sample['split'] == 'pretrain_val': # save HSV image for pretraining
        hsv_image = cv2.cvtColor(out_image.transpose(1, 2, 0), cv2.COLOR_RGB2HSV).transpose(2, 0, 1)
        
        target_path = os.path.join('data', 'splits', sample['split'], 'target', filename + '.tif')
        with rio.open(target_path, 'w', **out_meta) as dst:
            dst.write(hsv_image)

def extract_raster(samples_group):
    
    raster_path = samples_group[0]
    with rio.open(raster_path) as raster_dataset:
        samples_group[1].apply(lambda x: extract_mask(x, raster_dataset), axis=1)

n_threads = 8

# sample train, test and val splits first
# sub_samples = samples[samples['split'].isin(['train', 'test', 'val'])]
with ThreadPoolExecutor(max_workers=n_threads) as executor:
    list(executor.map(extract_raster, list(sub_samples.groupby('raster_path'))))


In [ ]:
# convert samples in train, test, val splits to png for annotation
for split in ['train', 'test', 'val']:
    
    os.makedirs(os.path.join('data', 'roboflow', split, 'input'), exist_ok=True)
    os.makedirs(os.path.join('data', 'roboflow', split, 'target'), exist_ok=True)
    samples_split = samples[samples['split'] == split]
    
    sample_files = glob(f'data/splits/{split}/input/*.tif')
    sample_ids = [int(os.path.basename(file).replace('.tif', '')) for file in sample_files]
    
    for id, file in zip(sample_ids, sample_files):
        with rio.open(file) as src:
            img = Image.fromarray(src.read().transpose(1, 2, 0))
            meta = src.meta
        
        # add lat, long of the centroid of the sample to the metadata
        sample = samples_split.loc[id]
        if type(sample) == gpd.GeoDataFrame:
            sample = sample.iloc[0]
        centroid = sample['geometry'].centroid
        meta['lat'] = centroid.y
        meta['lon'] = centroid.x
        
        # from histogram vector, add the relative frequency of each class to the metadata
        hist = sample['hist_vector']
        legend_classes = LEGEND_CLASSES.copy()
        legend_classes.pop(0) # remove nodata class
        
        meta['class_freq'] = {legend_classes[i+1]: hist[i] for i in range(len(hist))}

        # need to convert CRS to string such that it is serializable
        meta['crs'] = meta['crs'].to_string()
        
        with open(f'data/png_images/{split}/input/{id}.json', 'w') as f:
            json.dump(meta, f, indent=4)
        
        img.save(f'data/png_images/{split}/input/{id}.png')

In [ ]:
# sample pretrain and pretrain_val splits
n_threads = 16
sub_samples = samples[samples['split'].isin(['pretrain', 'pretrain_val'])]
with ThreadPoolExecutor(max_workers=n_threads) as executor:
    list(executor.map(extract_raster, list(sub_samples.groupby('raster_path')))
)

c:\Users\dh2306\projects\ms-land-cover\.env\Lib\site-packages\rasterio\features.py:392: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  _rasterize(valid_shapes, out, transform, all_touched, merge_alg)


In [ ]:
sampled_files = glob('data/splits/*/input/*.tif')
sampled_ids = [int(os.path.basename(file).replace('.tif', '')) for file in sampled_files]
remaining_samples = samples[~samples.index.isin(sampled_ids)]

display(remaining_samples.groupby('split').count())

,geometry,hist_vector,hist_vector_scaled,hist_vector_pca,cluster,index_right,raster_path,raster_geometry
split,,,,,,,,
pretrain,8,8,8,8,8,7,7,7


In [ ]:
# create segmentation masks using mean shift clustering
def segment_image(sample):
    
    filename = str(sample.name)
    input_path = os.path.join('data', 'splits', sample['split'], 'input', filename + '.tif')
    target_path = os.path.join('data', 'splits', sample['split'], 'masks', filename + '.gpkg')
    
    os.makedirs(os.path.dirname(target_path), exist_ok=True)
    
    with rio.open(input_path) as src:
        img = src.read().transpose(1, 2, 0)
        transform = src.transform
    
    color_radius = 20
    spatial_radius = 50
    i = 0
    while True:
        
        img = cv2.pyrMeanShiftFiltering(img, sr=color_radius, sp=spatial_radius).transpose(2, 0, 1)
        shapes = rio.features.shapes(img, mask=None, transform=transform)
        features = gpd.GeoDataFrame.from_features([{'geometry': shape, 'properties': {'class': None}} for shape, _ in shapes], crs=src.crs)
        i += 1
        
        if len(features) < 20000 or len(features) > 125000:
            break
        
        if i % 2 == 0:
            spatial_radius += 5
        else:
            color_radius += 5

    features.to_file(target_path, driver='GPKG')


sub_samples = samples[samples['split'].isin(['train', 'test', 'val'])].reset_index().drop_duplicates('index').set_index('index')

n_threads = 4
split_samples = np.array_split(sub_samples, n_threads)
with ThreadPoolExecutor(max_workers=n_threads) as executor:
    list(executor.map(lambda x: x.apply(segment_image, axis=1), split_samples))

c:\Users\dh2306\projects\ms-land-cover\.env\Lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: 'GeoDataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'GeoDataFrame.transpose' instead.
  return bound(*args, **kwds)
c:\Users\dh2306\projects\ms-land-cover\.env\Lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: 'GeoDataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'GeoDataFrame.transpose' instead.
  return bound(*args, **kwds)
c:\Users\dh2306\projects\ms-land-cover\.env\Lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: 'GeoDataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'GeoDataFrame.transpose' instead.
  return bound(*args, **kwds)
c:\Users\dh2306\projects\ms-land-cover\.env\Lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: 'GeoDataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'GeoDataFrame.tr

(256, 256, 3)
uint8
(256, 256, 3)
uint8
(256, 256, 3)
uint8
(256, 256, 3)
uint8
(256, 256, 3)
uint8


error: OpenCV(4.10.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\segmentation.cpp:368: error: (-210:Unsupported format or combination of formats) Only 8-bit, 3-channel images are supported in function 'cv::pyrMeanShiftFiltering'


<!-- ## Sampling Points for Accuracy Assesment -->

In [ ]:
sample_files = glob('data/splits/train/input/*.tif')
sample_files.extend(glob('data/splits/val/input/*.tif'))
sample_files.extend(glob('data/splits/test/input/*.tif'))

def cluster_image(sample_file, n_clusters=256):
    
    # print('A')
    
    with rio.open(sample_file) as src:
        data = src.read()
        meta = src.meta
    
    # print('B')
    
    data_old_shape = data.shape
    data = data.transpose(1, 2, 0).reshape(-1, 3)
    
    # print('C')
    
    pipeline = Pipeline([
        ('scaler', RobustScaler()),
        ('kmeans', KMeans(
            n_clusters=n_clusters, 
            n_init=10, 
            max_iter=300, 
            random_state=1701, 
        ))
    ])
    
    # print('D')
    
    data = pipeline.fit_transform(data)
    
    # print('E')

    data = data.reshape( 
        data_old_shape[1], 
        data_old_shape[2], 
        n_clusters
    )
    data = data.argmax(axis=-1).astype(np.uint8)
    
    out_filename = sample_file.replace('input', 'clusters').replace('.tif', '.gpkg')
    os.makedirs(os.path.dirname(out_filename), exist_ok=True)
    out_meta = meta.copy()
    out_meta['count'] = 1
    out_meta['dtype'] = 'uint8'
    shapes = rio.features.shapes(data, mask=None, transform=out_meta['transform'])
    features = gpd.GeoDataFrame.from_features([{'geometry': shape, 'properties': {'class': None}} for shape, _ in shapes], crs=out_meta['crs'])
    features.to_file(out_filename, driver='GPKG')

for sample_file in tqdm(sample_files, total=len(sample_files), desc='Clustering images', unit='image'):
    cluster_image(sample_file)

Clustering images:   6%|▌         | 62/1000 [11:36<2:55:31, 11.23s/image]


KeyboardInterrupt: 